# Explore MLHD+, Canonical Data Dump, and MusicBrainz formats

Stage 1 goal: look at the raw files directly before writing any loader code in
`src/songpsychic/data/`, so the parser matches what's actually in the files instead of
assumptions from the docs.

Run first:
- `python scripts/fetch_sample_data.py` (one MLHD+ partial shard ~2GB, one complete shard ~15GB)
- `python scripts/download_data.py --dataset canonical` (ID/name lookup table, small)
- `python scripts/download_data.py --dataset musicbrainz --files mbdump.tar.bz2 mbdump-derived.tar.bz2` (~8GB)

In [1]:
import sys
import tarfile
from pathlib import Path

import pandas as pd
import zstandard

sys.path.insert(0, "../src")
from songpsychic.data import canonical as canonical_data

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")

## MLHD+ shards

Each shard is a `.tar` of per-user files. Start by just listing what's inside — don't assume a
naming scheme or column layout yet.

In [2]:
partial_path = RAW / "mlhd" / "mlhdplus-partial-0.tar"

with tarfile.open(partial_path) as tar:
    partial_members = tar.getmembers()

# Tars also contain directory entries; extractfile() returns None for those, so keep regular files only
partial_files = [m for m in partial_members if m.isfile()]

print(f"{len(partial_members)} members ({len(partial_files)} files)")
for m in partial_members[:10]:
    print(m.name, m.size, "file" if m.isfile() else "dir/other")

37358 members (37342 files)
00 0 dir/other
00/004927e0-092d-4eea-b928-eea7a8ca3b48.txt.zst 60367 file
00/00174fbf-5e8e-461c-b2ad-95f9319a8fcf.txt.zst 55183 file
00/0004fbd1-1673-497f-bf3d-4a40e2acadd7.txt.zst 165096 file
00/0082cb10-4630-4148-9342-f22c4b3f6d2b.txt.zst 55778 file
00/0002f253-70c5-4fd0-9578-ab3d8df6c5a4.txt.zst 18772 file
00/0026de04-67f6-4f80-aa5e-5dd504453a82.txt.zst 79043 file
00/00721df7-61fe-4329-86b2-6d6c401b4b95.txt.zst 48900 file
00/0052102e-c1ab-4e7f-aa16-efa6720b7926.txt.zst 27575 file
00/00d8548f-fd8a-400d-8601-4a0ca35a89b5.txt.zst 26323 file


In [3]:
# Each per-user file is ALSO zstd-compressed (note the .txt.zst extension) — decompress before reading
with tarfile.open(partial_path) as tar:
    with tar.extractfile(partial_files[0]) as f:
        with zstandard.ZstdDecompressor().stream_reader(f) as reader:
            raw = reader.read(2000)

print(raw.decode("utf-8", errors="replace"))

1159396384	06fb1c8b-566e-4cb2-985b-b467c90781d4	18519193-9e7c-4515-baab-bed88265a51d	
1160103698	618b6900-0618-4f1e-b835-bccb17f84294		
1160106006	e21857d5-3256-4547-afb3-4b6ded592596		
1160106860	c296e10c-110a-4103-9e77-47bfebb7fb2e		
1160108092	83d91898-7763-47d7-b03b-b92132375c47	54550ba2-5a37-3164-b14d-d07189cb2ed6	
1160115712	382f1005-e9ab-4684-afd4-0bdae4ee37f2	105029c7-f9c6-4009-99ef-3649ee2f9657	
1160153286	22c48674-6d6c-43a4-a8d8-300a9185f197		
1160156551			
1160157006	83d91898-7763-47d7-b03b-b92132375c47	54550ba2-5a37-3164-b14d-d07189cb2ed6	
1160157725	83d91898-7763-47d7-b03b-b92132375c47	54550ba2-5a37-3164-b14d-d07189cb2ed6	
1160158468	12327d75-47d5-45d9-84c2-3760b9210c17		
1160161430	ece57992-dc2e-4f67-a269-fa43626c1a3d	b19eb9fb-0bac-41c0-b585-0a9348ac0f33	
1160236505	ece57992-dc2e-4f67-a269-fa43626c1a3d	b19eb9fb-0bac-41c0-b585-0a9348ac0f33	
1160237902	2d196645-cab3-430e-88f6-bd57d95940ea		
1160241226	c296e10c-110a-4103-9e77-47bfebb7fb2e		
1160249612	ca39d50f-9885-420e-88d6

## Compare against the complete shard

"Partial" vs "complete" turns out to be about MBID *resolution*, not listen volume: partial-shard
rows are frequently missing the trailing recording MBID (unresolved), complete-shard rows almost
always have all 4 fields populated.

In [4]:
complete_path = RAW / "mlhd" / "mlhdplus-complete-0.tar"

with tarfile.open(complete_path) as tar:
    complete_members = tar.getmembers()
    complete_files = [m for m in complete_members if m.isfile()]
    with tar.extractfile(complete_files[0]) as f:
        with zstandard.ZstdDecompressor().stream_reader(f) as reader:
            complete_raw = reader.read(2000)

print(f"{len(complete_members)} members ({len(complete_files)} files)")
print(complete_raw.decode("utf-8", errors="replace"))

37358 members (37342 files)
1160100311	4d5447d7-c61c-4120-ba1b-d7f471d385b9	56d2ee1e-8a89-4ecc-a101-736969d15c49	0134acbe-a16b-4836-bcb6-473c04ecc8f2
1160100511	65f4f0c5-ef9e-490c-aee3-909e7ae6b2ab	2529f558-970b-33d2-a42c-41ab15a970c6	fe4a0608-cf7f-444a-a932-a1b77c766673
1160100860	6faa7ca7-0d99-4a5e-bfa6-1fd5037520c6	4312b6ca-cf74-46cb-9bfb-d4472de3677c	f4613de4-f4e8-4d8b-b682-26d50676755d
1160101209	83d91898-7763-47d7-b03b-b92132375c47	8eb1532a-bccb-4cc4-b590-5be19386a6f7	2423ca5e-99d3-4b49-be53-542e6ee38a82
1160101564	0039c7ae-e1a7-4a7d-9b49-0cbc716821a6	33be482c-0f66-4dd5-9ecd-55918565c6c1	a5741d3d-c925-4070-8a83-910d23eb33d4
1160101754	bea235a0-db63-44b1-881e-264d25791f5a	6d07faa4-8937-35d5-bbca-5093f068bb4d	64919dfd-23de-452e-ad5b-a6b2a9497c91
1160101939	3a54bffa-2314-44a2-927b-60144119c780	7fdf33ff-9dc7-41c4-bbfe-c5c7d1f68751	b7fa4a61-2446-40ea-b2d2-a7802a215134
1160102215	a3cb23fc-acd3-4ce0-8f36-1e5aa6a18432	59f2d05c-421b-473f-941a-02752f6e779e	809e8321-9e38-4ebf-b467-1cb6d85f5

## Canonical Data Dump

Needs `python scripts/download_data.py --dataset canonical` first. The archive is `.tar.zst`.
Extracting goes to `data/interim/`, not `data/raw/` — raw stays exactly as downloaded; the
decompressed CSVs are a derived, regeneratable-from-raw intermediate.

In [5]:
canonical_dir = RAW / "canonical"
archive = next(canonical_dir.glob("*.tar.zst"))
extracted_dir = INTERIM / "canonical"

canonical_data.extract_canonical_dump(archive, extracted_dir)

csv_paths = list(extracted_dir.glob("*.csv"))
csv_paths

[PosixPath('../data/interim/canonical/canonical_recording_redirect.csv'),
 PosixPath('../data/interim/canonical/canonical_musicbrainz_data.csv'),
 PosixPath('../data/interim/canonical/canonical_release_redirect.csv')]

In [6]:
for csv_path in csv_paths:
    print("---", csv_path.name, "---")
    display(pd.read_csv(csv_path, nrows=5))

--- canonical_recording_redirect.csv ---


,recording_mbid,canonical_recording_mbid,canonical_release_mbid
0,b80391be-803d-4ee6-bcce-837479a0fc0a,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8
1,3bd7729a-aafa-4ea1-863a-824c6391b596,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8
2,22760537-0a77-41d7-88ec-0e935e58677a,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8
3,c46d2015-cc39-4632-8b32-fa14d8372d65,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8
4,81dd4e68-ad37-4f7c-809e-a13d83ff9159,ecb125d7-d23e-4d76-8282-745713563110,ffb4aba3-8aa4-479d-9a62-1bbe881804b8


--- canonical_musicbrainz_data.csv ---


,id,artist_credit_id,artist_mbids,artist_credit_name,release_mbid,release_name,recording_mbid,recording_name,combined_lookup,score
0,1,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,432d5329-fdbb-4803-9f74-ca2d71118ab1,Songs of Quigley's Village Volume 2,5c29abbd-b602-4f22-8c40-bc119abc86c9,Bedtime Prayer,variousartistsbedtimeprayer,87910
1,2,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,432d5329-fdbb-4803-9f74-ca2d71118ab1,Songs of Quigley's Village Volume 2,c6839c82-bf0a-43a5-80ce-25d2b7f973f7,Quigley’s Village Theme,variousartistsquigleysvillagetheme,87910
2,3,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,61efbf58-5be4-40ba-9bef-49c92cc5b8ca,Sofé Linzy,7b1f193b-c9ba-48d6-bd6f-8afd02d489a2,Pot Pourri Sega,variousartistspotpourrisega,188739
3,4,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,82b37a32-e983-4b6b-b353-f38b9018ea7f,滾石民歌時代百大經典,4cbd4a88-eff8-47e5-8d18-683b1cdff891,乡愁四韵,variousartistsXiang Chou Si Yun,191119
4,5,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,82b37a32-e983-4b6b-b353-f38b9018ea7f,滾石民歌時代百大經典,77c52b21-0ce0-4c49-a99d-d25ebd4f5f71,白霏霏,variousartistsBai Fei Fei,191119


--- canonical_release_redirect.csv ---


,release_mbid,canonical_release_mbid,release_group_mbid
0,b3577744-e721-4357-bc60-d79c02863dac,b3577744-e721-4357-bc60-d79c02863dac,e8bee759-9efc-35c2-93d7-09ace9123467
1,96ccdd6e-ef20-32b2-ac99-8840c1a191c1,3a25785e-50ec-383c-89cf-fd512181449c,8b6f133a-2fdf-3cc2-b84d-1c889adc0939
2,1a1befee-6d0c-4d09-a150-788010080911,3a25785e-50ec-383c-89cf-fd512181449c,8b6f133a-2fdf-3cc2-b84d-1c889adc0939
3,9af8d45c-d4df-310d-baff-e5b7d956c2c8,3a25785e-50ec-383c-89cf-fd512181449c,8b6f133a-2fdf-3cc2-b84d-1c889adc0939
4,a0dc813b-ff05-4e47-b825-7cf8311e5052,3a25785e-50ec-383c-89cf-fd512181449c,8b6f133a-2fdf-3cc2-b84d-1c889adc0939


## MusicBrainz database dump (core + derived)

Needs `python scripts/download_data.py --dataset musicbrainz --files mbdump.tar.bz2
mbdump-derived.tar.bz2` first. This is PostgreSQL's plain-text `COPY` format: one file per table,
tab-separated, `\N` for null, **no header row and no column names** — those live in the
MusicBrainz server schema, not in the dump itself.

In [7]:
mb_core_path = RAW / "musicbrainz" / "mbdump.tar.bz2"

# bz2 has no random access — getmembers() would decompress the WHOLE ~7.6GB file just to list
# names. Stream instead and stop as soon as we have enough to look at.
mb_core_preview = []
with tarfile.open(mb_core_path, mode="r|bz2") as tar:
    for i, member in enumerate(tar):
        mb_core_preview.append(member)
        if i >= 19:
            break

print("first 20 members (streamed — getting an exact total would mean decompressing everything):")
for m in mb_core_preview:
    print(m.name, m.size)

first 20 members (streamed — getting an exact total would mean decompressing everything):
TIMESTAMP 30
COPYING 6390
README 213
REPLICATION_SEQUENCE 7
SCHEMA_SEQUENCE 3
mbdump/alternative_release_type 187
mbdump/area 13179740
mbdump/area_alias 6576521
mbdump/area_alias_type 175
mbdump/area_gid_redirect 13923
mbdump/area_type 1870
mbdump/artist 444428362
mbdump/artist_alias 58689342
mbdump/artist_alias_type 176
mbdump/artist_credit 425295604
mbdump/artist_credit_gid_redirect 6019935
mbdump/artist_credit_name 244148067
mbdump/artist_gid_redirect 6710135
mbdump/artist_ipi 6046936
mbdump/artist_isni 9588383


In [8]:
# Find the `artist` table specifically — stream and stop as soon as we reach it, rather than
# decompressing the whole archive to build a full member list first
artist_raw = None
with tarfile.open(mb_core_path, mode="r|bz2") as tar:
    for member in tar:
        if member.name.endswith("/artist"):
            with tar.extractfile(member) as f:
                artist_raw = f.read(2000)
            break

print(artist_raw.decode("utf-8", errors="replace"))

2252039	fadeb38c-833f-40bc-9d8c-a6383b38b1be	Доктор Сатана	Доктор Сатана	\N	\N	\N	\N	\N	\N	\N	\N	\N		0	2021-11-23 07:08:52.479537+00	f	\N	\N
371203	49add228-eac5-4de8-836c-d75cde7369c3	Pete Moutso	Moutso, Pete	\N	\N	\N	\N	\N	\N	1	\N	\N		0	\N	f	\N	\N
3087346	dfdce491-133d-4e9f-9e48-795587e181b0	UNlT	UNlT	\N	\N	\N	\N	\N	\N	\N	\N	\N		0	2025-09-14 23:53:28.004798+00	f	\N	\N
2851271	165a49a0-2b3b-4078-a3c1-905afdc07c0a	Babyglock	Babyglock	\N	\N	\N	\N	\N	\N	\N	\N	\N		0	2024-10-19 03:33:55.474151+00	f	\N	\N
145773	7b4a548e-a01a-49b7-82e7-b49efeb9732c	Aric Leavitt	Leavitt, Aric	\N	\N	\N	\N	\N	\N	\N	\N	\N		0	\N	f	\N	\N
1076328	60aca66f-e91a-4cb5-9308-b6e293cd833e	Fonograff	Fonograff	\N	\N	\N	\N	\N	\N	\N	\N	\N		0	2014-01-10 16:25:20.992213+00	f	\N	\N
1172876	3e1bd546-d2a7-49cb-b38d-d70904a1d719	Al Street	Street, Al	\N	\N	\N	\N	\N	\N	1	222	1		0	2014-11-23 14:07:19.782509+00	f	\N	\N
220155	df120895-f6c6-4a66-b9cf-73350f0beb61	Love .45	Love .45	\N	\N	\N	\N	\N	\N	\N	\N	\N		0	\N	f	\N	\N
618464	c14f8d

In [9]:
first_row = artist_raw.decode("utf-8", errors="replace").splitlines()[0]
print(f"{first_row.count(chr(9)) + 1} tab-separated columns")
print(first_row)

19 tab-separated columns
2252039	fadeb38c-833f-40bc-9d8c-a6383b38b1be	Доктор Сатана	Доктор Сатана	\N	\N	\N	\N	\N	\N	\N	\N	\N		0	2021-11-23 07:08:52.479537+00	f	\N	\N


## Derived dump (tags, ratings, search indexes)

Same `COPY` format as the core dump, but licensed CC BY-NC-SA (vs. CC0 for the core dump) — keep
the two separate if this data is ever reused outside this project.

In [10]:
mb_derived_path = RAW / "musicbrainz" / "mbdump-derived.tar.bz2"

mb_derived_preview = []
with tarfile.open(mb_derived_path, mode="r|bz2") as tar:
    for i, member in enumerate(tar):
        mb_derived_preview.append(member)
        if i >= 19:
            break

print("first 20 members (streamed, same reason as the core dump above):")
for m in mb_derived_preview:
    print(m.name, m.size)

first 20 members (streamed, same reason as the core dump above):
TIMESTAMP 30
COPYING 15818
README 213
REPLICATION_SEQUENCE 7
SCHEMA_SEQUENCE 3
mbdump/annotation 669253233
mbdump/area_annotation 133
mbdump/area_tag 155794
mbdump/artist_annotation 704751
mbdump/artist_meta 41499901
mbdump/artist_tag 34168155
mbdump/event_annotation 85999
mbdump/event_meta 2455786
mbdump/event_tag 625187
mbdump/instrument_annotation 1060
mbdump/instrument_tag 177228
mbdump/label_annotation 166487
mbdump/label_meta 4500748
mbdump/label_tag 1719747
mbdump/medium_index 275794909


No column names ship with either dump — table schemas live in the MusicBrainz server repo's SQL:
https://github.com/metabrainz/musicbrainz-server/blob/master/admin/sql/CreateTables.sql.
Needed before writing a real loader in `src/songpsychic/data/`.

## Notes

**MLHD+ per-listen format** (tab-separated, no header): `timestamp` (unix epoch seconds),
`artist_mbid`, `release_mbid`, `recording_mbid`. The last field is frequently empty in the
*partial* shard (unresolved recording) and almost always present in the *complete* shard — that's
the actual partial/complete distinction, not listen count.

**Member counts**: both sampled shards (`-0`) had 37358 tar members / 37342 regular files (16
shards means per-user files are bucketed by hash prefix, not by user volume).

**Canonical CSV → MLHD+ join**: `canonical_recording_redirect.csv` (`recording_mbid` →
`canonical_recording_mbid`) is what collapses an MLHD+ row's `recording_mbid` onto one canonical
ID. `canonical_musicbrainz_data.csv` then gives that canonical recording's `recording_name` /
`artist_credit_name` directly — no need to touch the full MusicBrainz dump just for display names.

**MusicBrainz `artist` table**: 19 tab-separated columns, matching the MusicBrainz schema order:
`id, gid, name, sort_name, begin_date_{year,month,day}, end_date_{year,month,day}, type, area,
gender, comment, edits_pending, last_updated, ended, begin_area, end_area`. `gid` is the MBID used
everywhere else (in canonical CSVs, in MLHD+). No header row, confirmed — relying on the upstream
schema SQL (`CreateTables.sql`) is required, not optional.

**Stage 3 candidates seen in `mbdump-derived.tar.bz2`'s first 20 members**: `artist_tag` (34MB) —
exactly the genre/tag signal stage 3 needs. Haven't scanned far enough to find the `tag` table
itself (the id → tag-name lookup) yet — next step if genre features are needed.

**Surprising**: `mbdump/annotation` alone is ~669MB (free-text edit annotations, not useful for
us) — a reminder that these dumps contain a lot of MusicBrainz-internal bookkeeping tables mixed
in with the ones we actually want (artist, artist_credit_name, artist_tag, tag).